Currently, the working version of both the average and max pooling backward passes suffer from major GPU bottleneck issues:

* Max: There are two main issues with the implementation. The first, we broadcast the axes into four seperate 4D arrays taking up four times the memory space. The second issue is that using `cp.add.at`. This method is designed to handle cases where multiple indices point to the exaxct same memory address, but because we are using this atomic operation, the GPU has to introduce a lock at a memory address serilaizing the execution. 

* Average: There are three main issues with the elementwise kernel appraoch. 
    1. We used `atomicAdd(...)` locking a specific memory address such that only a single thread is able to update its space at a time. Average pooling will distribute gradients to every single pixel inside a pooling window. If our windows overlap slightly (e.g., $3 \times 3$ filters with a stride of $2$), multiple threads will simultaneously compute the exact same `target_idx`.

    2. GPUs read data from VRAM in wide, contiguous blocks(with sizes of 32-byte or 128-byte segments). The GPU will use a hardware scheduling unit (AMD Wavefront, or Nvidia Warp) will send a **single memory transaction**, sending all the byte information needed for each thread to read at the same time. For example, if we have 32 threads that would like to read data to perform our average pooling calculation, then we'll send a 128 byte block of data (4 bytes per thread) in a single transaction. **However**, inside the code the line appears:
    ```python
    int target_idx = ((s * H_in + h_in) * W_in + w_in) * C + c;
    ```
    Because the kernel is launched over `dvalues.ravel()`, each thread will handle a different output position. In other words, instead of us cleanly sending a uniform block of memory in a single sweep, the threads are scatering data all over the place, forcing the GPU to repeatedly send fetch instructions.

    3. Misuse of `cp.ElementwiseKernel` for a scatter operation. 
    In this case, `ElementwiseKernel` is best use under the assumption of **element-to-elemnt mapping** (e.g., Thread $i$ reads input $i$ and writes cleanly to output $i$)
    Here, we used `raw float32 dinputs` and wrote a custon kernel to handle the scattering and unpooling operations. However, because we didn't follow the simple rule the elementwise kernel is intended for, we cannot optimize memory block loading or shared cache for loops it doesn't know exists. 

Below is the old pooling layer with its backwards pass:

In [ ]:
import cupy as cp
from cupy.lib.stride_tricks import as_strided
scatter_avg_pooling_kernel = cp.ElementwiseKernel(
    in_params='''
        float32 dval,
        int32 S, int32 H_out, int32 W_out, int32 C,
        int32 H_in, int32 W_in,
        int32 fH, int32 fW,
        int32 sH, int32 sW
    ''',
    out_params='raw float32 dinputs',
    operation=r'''
        // i is the linear index into dvalues
        // Decode: (s, h_out, w_out, c)
        int c = i % C;
        int w_out = (i / C) % W_out;
        int h_out = (i / (C * W_out)) % H_out;
        int s = i / (C * W_out * H_out);
        
        // Gradient to distribute to each position in the pool
        float grad_per_position = dval / (fH * fW);
        
        // Calculate starting position in input
        int h_start = h_out * sH;
        int w_start = w_out * sW;
        
        // Distribute gradient to all positions in this pool window
        for (int fh = 0; fh < fH; fh++) {
            for (int fw = 0; fw < fW; fw++) {
                int h_in = h_start + fh;
                int w_in = w_start + fw;
                
                // Calculate linear index in dinputs
                int target_idx = ((s * H_in + h_in) * W_in + w_in) * C + c;
                
                // Atomic add to handle overlapping windows
                atomicAdd(&dinputs[target_idx], grad_per_position);
            }
        }
    ''',
    name='scatter_avg_pooling'
)

class Pooling:

    def __init__(self, filter_size = (2, 2), strides = (2, 2),
                  padding = "valid", pooling_type = "max"):
        self.filter_size = filter_size
        self.strides = strides
        self.padding = padding
        self.pooling_type = pooling_type
        
    def forward(self, inputs, training):
        #Inputs should be of shape (S, H_in, W_in, C = D_in) 
        inputs = inputs.astype(cp.float32, copy = False)
        if inputs.ndim != 4:
            raise ValueError(f"Expected a 4D tensor, got {inputs.ndim} instead.")
        S, H_in, W_in, C = inputs.shape
        fH, fW = self.filter_size
        sH, sW = self.strides

        padding = self.padding
        if padding == "valid":
            H_out = int(cp.floor((H_in - fH) / sH + 1).item())
            W_out = int(cp.floor((W_in - fW) / sW + 1).item())

            self.pad_top, self.pad_bottom, self.pad_left, self.pad_right = 0, 0, 0, 0
        elif padding == "same":
            
            H_out = int(cp.ceil(H_in / sH).item())
            W_out = int(cp.ceil(W_in / sW).item())

            pad_h = max((H_out - 1) * sH + fH - H_in, 0)
            pad_w = max((W_out - 1) * sW + fW - W_in, 0)
            self.pad_top = pad_h // 2
            self.pad_bottom = pad_h - self.pad_top
            self.pad_left = pad_w // 2
            self.pad_right = pad_w - self.pad_left
            inputs = cp.pad(inputs, ((0,0), (self.pad_top,self.pad_bottom), (self.pad_left,self.pad_right), (0,0)), mode='constant')
        else: 
            raise ValueError(f"Expected padding == valid or same, recieved {padding} instead")

        #cast our output dimensions into ints from floats. 
        H_out, W_out = int(H_out), int(W_out)

        #create output tensor with new sizes
        self.output = cp.zeros(shape = (S, H_out, W_out, C))
        self.inputs = inputs
        patches = as_strided(
            inputs,
            shape = (S, H_out, W_out, fH, fW, C), 
            strides = (
                inputs.strides[0],      #step between samples
                inputs.strides[1] * sH, #step between rows
                inputs.strides[2] * sW, #step between columns
                inputs.strides[1],      #Move down 1 row inside patch
                inputs.strides[2],      #move right 1col inside patch
                inputs.strides[3],      #step between each channel
            )
        )

        if self.pooling_type == "max":
            pooled = patches.max(axis = (3, 4)) 
            #We'll reshape the window to become a 1d array of size fH * fW
            patches_reshaped = patches.reshape(S, H_out, W_out, fH * fW, C)
            flat_indicies = patches_reshaped.argmax(axis = 3)

            #Now, we'll convert those flat indicies back to row col coordinates withing each
            #(fH, fW) patch
            max_rows, max_cols = cp.unravel_index(flat_indicies, (fH, fW)) 
            self.max_indicies = (max_rows, max_cols) 
        
        elif self.pooling_type == "average":
            pooled = patches.mean(axis = (3, 4))
            
        #Store both of these for backprop
        self.inputs = inputs
        self.output = pooled
        return self.output

    def backward(self, dvalues):
        
        dvalues = dvalues.astype(cp.float32, copy = False)
        #We want the same shape as self.inputs, we'll populate the tensor with zeros at first then unpool later.
        self.dinputs = cp.zeros_like(self.inputs, dtype=cp.float32)
        S, H_out, W_out, C = dvalues.shape
        H_in, W_in = self.inputs.shape[1:3]
        fH, fW = self.filter_size
        sH, sW = self.strides
        
        if self.pooling_type == "max":
            max_rows, max_cols = self.max_indicies
            
            s_idx = cp.arange(S)[:, None, None, None]      # Shape: (S, 1, 1, 1)
            h_idx = cp.arange(H_out)[None, :, None, None]  # Shape: (1, H_out, 1, 1)
            w_idx = cp.arange(W_out)[None, None, :, None]  # Shape: (1, 1, W_out, 1)
            c_idx = cp.arange(C)[None, None, None, :]      # Shape: (1, 1, 1, C)
            
            # Calculate where in the input each gradient should go
            # Broadcasting creates arrays of shape (S, H_out, W_out, C)
            input_h = h_idx * sH + max_rows  # h_idx broadcasts, max_rows is already (S, H_out, W_out, C)
            input_w = w_idx * sW + max_cols
            
            # Accumulate gradients at the right positions
            # cp.add.at handles if multiple output positions map to same input position
            cp.add.at(self.dinputs, (s_idx, input_h, input_w, c_idx), dvalues)
        
        elif self.pooling_type == "average":
            
            scatter_avg_pooling_kernel(
                dvalues.ravel(),
                S, H_out, W_out, C,
                H_in, W_in, 
                fH, fW,
                sH, sW,
                self.dinputs.ravel()
            )
        return self.dinputs

# How do we Optimize our Backwards Pass? 

 Well for one, we'll do away with a custom kernel for our project as we're able to use two routes to perform the backwards pass of our pooling layer. The first is under the assumption that our filter sizes $(fH, fW)$ matches our stride height and width $(sH, sW)$. We'll still account for the average and max versions of the code, but we'll divide each approach into two different implementations, where our second approach will be a more generalized version that supports any filter and stride dimensions at the cost of computational speed. 

Inside `Backpropagation_Pooling`, there includes theory and implementation of both the max and average pooling backward passes. This notebook will be used as a continuation of that notebook, where we'll introduce the idea and implementation of these faster methods. 

# Max Pooling

In max pooling we'll recieve the gradient `dvalues` which is our $\frac{\partial y}{\partial L}$ which is our forward pass outputs values wrt. the loss function. We need to arrive at `dinputs` or $\frac{\partial x}{\partial L}$. An expanded form of our backwards pass is written below:

$$
\nabla_X L_{s, h', w', c} = \sum_{h, w} \nabla_Y L_{s, h, w, c} \cdot \mathbb{I}\left( X_{s, h', w', c} = Y_{s, h, w, c} \right)
$$

where
* $\nabla_X L_{s, h', w', c}$: The `dinputs` or downstream gradient wrt. the loss function. Our batch elements and channels never change during pooling, meaning the layout matches the original input shape $(S, H_{in}, W_{in}, C)$. $h'$ is a specific row index inside the range of `H_in`, and $w`$ is a specific column index inside the range of `W_in`. 

* $\sum {h, w}$: Evaluates a summation across all sliding window spatial coordinates $(h, w)$

* $\nabla_Y L_{s, h, w, c}$: Our `dvalues` or upstream gradient wrt. the loss function. $h$ is a specific row index inside the range of `H_out`, and $w$ is a specific column index inside the range of `W_out`. 

* $\mathbb{I}\left( X_{s, h', w', c} = Y_{s, h, w, c} \right)$: Here, we verify whether the original input pixel value $X$ at position $(h', w')$ matches the pooled maximum output value $Y$ extracted from its local window. When they match, the upstream gradient value is routed entirely through to this single index. 

This last part is the most informative, if we hit a position that contained a max value, then we have a position that influenced the loss function at input pixel $X$. Otherwise, if the condition is false the indicator function evaluates to 0, meaning that we have hit a point that did not influence the position of the loss function. 

EX: Lets assume we're using a input shape $X: 4\times 4$, a filter size $2\times2$ along with a stride of $2\times2$. Since there are non overlapping windows, we can do the *fast* path. 

$$ X = \begin{bmatrix} 
1 & 2 & 3 & 1 \\
0 & 1 & 2 & 1 \\
4 & 2 & 1 & 0 \\
3 & 1 & 0 & 5
\end{bmatrix}
\implies
Y = \begin{bmatrix}
2 & 3 \\
4 & 5
\end{bmatrix}
$$

Our forward pass output has shape $H_{out}, W_{out}$ or $2\times2$. During our backwards pass our upstream gradient of the pooling layer will have that same shape $2\times2$. The actual values will of course be different, lets continue. 
$$\text{dvalues } (\nabla_Y L) = \begin{bmatrix}
1 & 9 \\
1 & 1
\end{bmatrix}$$

$$\text{Indicator Mask } \mathbb{I} = \begin{bmatrix}
0 & 1 & 1 & 0 \\
0 & 0 & 0 & 0 \\
1 & 0 & 0 & 0 \\
0 & 0 & 0 & 1
\end{bmatrix}$$

Now that we have the indicator mask, we want to broadcast the max value (which is now $\text{dvalues}_{h,w})$ to the indicator mask, after broadcasting we'd like to perform elementwise multiplication and arrive at our `dinputs`:

$$ \text{dinputs } (\nabla_X L) = \begin{bmatrix}
0 \times 1 & 1 \times 1 & 1 \times 9 & 0 \times 9 \\
0 \times 1 & 0 \times 1 & 0 \times 9 & 0 \times 9 \\
1 \times 1 & 0 \times 1 & 0 \times 1 & 0 \times 1 \\
0 \times 1 & 0 \times 1 & 0 \times 1 & 1 \times 1
\end{bmatrix} $$
$$\text{dinputs } (\nabla_X L) = \begin{bmatrix}
0 & \mathbf{1} & \mathbf{9} & 0 \\
0 & 0 & 0 & 0 \\
\mathbf{1} & 0 & 0 & 0 \\
0 & 0 & 0 & \mathbf{1}
\end{bmatrix}$$

To really quickly summarize the algorithm:
* Step 1: Recieve `dvalues`
* Step 2: Generate the binary indicator mask 
* Step 3: Expand `dvalues` and perform elementwise multiplication against the binary mask
* Step 4: Fold/Accumulate `dpatcheds` back into the full-size `dinputs` canvas layout. 

## Implementing our *Fast* Max Pooling Pass: 


If you remember, we used `self.patches` to step through each image and step through by whatever the stride size is. We'll borrow this code from our convolution layer. Therefore, this path will be under the assumption of the following:
```python

if self.pooling_type == "max":
    if self.strides == self.filter_size:
        #compute non-overlapping windows
        inputs = self.inputs
        patches = as_strided(
            inputs,
            shape = (S, H_out, W_out, fH, fW, C), 
            strides = (
                inputs.strides[0],      # Step between samples
                inputs.strides[1] * sH, # Step between rows
                inputs.strides[2] * sW, # Step between columns
                inputs.strides[1],      # Move down 1 row inside patch
                inputs.strides[2],      # Move right 1 col inside patch
                inputs.strides[3],      # Step between each channel
            )
        )
```

Now that we have our patches, we want to create the binary mask, that means wherever our max values were inside the stride. To do this, we can use our "output" values from the forward pass as this gives us the actual values. Then, wherever the "output" values equal the "input" values from the forward pass, then that means we can create a mask. The problem with this is `self.output` has shape $(S, H_{out}, W_{out}, C)$ and not this 6d array. This is a simple fix though, we can create a new axes to match the shape of our patches memory view. 

```python 
expanded_output = self.output[:, :, :, cp.newaxis, cp.newaxis, :]
mask = (patches == expanded_output)
```
Now that we have the mask, we want to broadcast the values from `dvalues` and perform elementwise multiplication into the mask. In cupy, the broadcasting operation is done via the asterik symbol *, we can do the following again

Mask Tensor: $(S, H_{out}, W_{out}, fH, fW, C)$
dvalues Tensor: $(S, H_{out}, W_{out}, 1, 1, C)$ (add dummy axis)
New Tensor: $(S, H_{out}, W_{out}, fH, fW, C)$

Under the hood we'll execute `mask * dvalues`, CuPy will create a kernel under the hood. The GPU threads will read a single value from `dvalues` and apply it across the entire `fH x fW` axes, therefore we broadcast the result from dvalues onto the mask. 

```python
dvalues_reshaped = dvalues[:, :, :, cp.newaxis, cp.newaxis, :]
dpatches = mask * dvalues_reshaped
```
Before we continue in implementing our backwards pass, we still have to account for padding assuming `self.padding = "same"`. We'll have to compute the padding from the forward pass. We'll cache these values `self.pad_top`, `self.pad_bottom`, `self.pad_left`, and `self.pad_right` and create a tensor `dinputs_padded` using these values. For our padded height and width that we'd like to retrieve, we can sum up the values that make up the height and width axes, then create the tensor of zeros. 

```python
pad_top = self.pad_top
pad_bottom = self.pad_bottom
pad_left = self.pad_left
pad_right = self.pad_right

padded_H = H_in + pad_top + pad_bottom
padded_W = W_in + pad_left + pad_right
dinputs_padded = cp.zeros((S, padded_H, padded_W, C), dtype=dvalues.dtype)
```


While it is nice that we have created the right values, we are still in 6D. We'd like to fold across the filter dimensions back into our 4d shape $(S, H_{out} * fH, W_{out} * fW, C)$. When we call `.reshape()`, our memory is laid out in C-contiguous order (row major), so our array is unrolled from the last dimension to the first dimension. When we perform reshape the framework will flatten out the window coordinates $(fH, fW)$ before moving onto the next window block $(W_out)$. We can reorder our axes first by using `.transpose()`. This will unroll our block rows with our filter row, and unroll our block columns with the filter columns. 

```python
dinputs_canvas = dpatches.transpose(0, 1, 3, 2, 4, 5).reshape(S, H_out * fH, W_out * fW, C)
```

Finally, we're at the right shape, but we need to account for when a input image is not perfectly divisible by our stride. Ex: $5\times5$ image with filter and stride of $2\times2$. We'll see each row except for the last column and the last row, to fix this we'll use the tensor `dinputs_padded`, then copy the gradients onto this final tensor. After this step we'll remove the fake padding borders and output our true `dinputs` tensor. 

```python
dinputs_padded[:, :H_out * fH, :W_out * fW, :] = dinputs_canvas
dinputs = dinputs_padded[:, pad_top : pad_top + H_in, pad_left : pad_left + W_in, :]
```

In [ ]:
def backward(self, dvalues):
    
    S, H_out, W_out, C = dvalues.shape
    fH, fW = self.filter_size
    sH, sW = self.strides
    _, H_in, W_in, _ = self.inputs.shape
    if self.pooling_type == "max":
        if self.strides == self.filter_size:


            if self.padding == "valid":
                pad_top = pad_bottom = pad_left = pad_right = 0
                padded_H, padded_W = H_in, W_in
            elif self.padding == "same":
                    
                pad_top = self.pad_top
                pad_bottom = self.pad_bottom
                pad_left = self.pad_left
                pad_right = self.pad_right

                padded_H = H_in + pad_top + pad_bottom
                padded_W = W_in + pad_left + pad_right

            dinputs_padded = cp.zeros((S, padded_H, padded_W, C), dtype=dvalues.dtype)
            #compute non-overlapping windows
            inputs = self.inputs
            patches = as_strided(
                inputs,
                shape = (S, H_out, W_out, fH, fW, C), 
                strides = (
                    inputs.strides[0],      # Step between samples
                    inputs.strides[1] * sH, # Step between rows
                    inputs.strides[2] * sW, # Step between columns
                    inputs.strides[1],      # Move down 1 row inside patch
                    inputs.strides[2],      # Move right 1 col inside patch
                    inputs.strides[3],      # Step between each channel
                )
            )
            expanded_output = self.output[:, :, :, cp.newaxis, cp.newaxis, :]
            mask = (patches == expanded_output)

            #expand upstream gradient 
            expanded_dvalues = dvalues[:, :, :, cp.newaxis, cp.newaxis, :]
            dpatches = expanded_dvalues * mask

            dinputs_canvas = dpatches.transpose(0, 1, 3, 2, 4, 5).reshape(S, H_out * fH, W_out * fW, C)
            
            dinputs_padded[:, :H_out * fH, :W_out * fW, :] = dinputs_canvas
            dinputs = dinputs_padded[:, pad_top : pad_top + H_in, pad_left : pad_left + W_in, :]
        if self.strides != self.filter_size:
            pass
        
        return dinputs
    if self.pooling_type == "average":
        pass
    return dinputs

# Average Pooling

First lets worry about the case of a *fast* average pooling layer where both the strides and filter dimensions are equal. If this is the case, we can incorporate a pretty similar stratagy of performing a mask. For this, we'll populate a 6D tensor with the same value. This value will be calculated from $1 / (fH * fW)$. . We can write an average pooling equation for our problem:

$$\nabla_X L_{s, h', w', c} = \sum_{h, w} \nabla_Y L_{s, h, w, c} \cdot \frac{1}{f_H \cdot f_W} \cdot \mathbb{I}\left( (h', w') \in \Omega_{h, w} \right)$$

where:
* $\nabla_X L_{s, h', w', c}$: The `dinputs` or downstream gradient wrt. the loss function. $h'$ is a specific row index inside the range of $H_in$, and $w'$ is a specific column index inside the range of $W_in$.

* $\sum _{h,w}$: Evaluates the summation across all sliding window spatial coordinates $(h, w)$. In both max and average pooling, the summation is necessary when the stride and filter dimensions are not the same. 

* $\nabla_Y L_{s, h, w, c}$: Our `dvalues` or upstream gradient wrt. the loss function. $h$ is a specific row index inside the range of `H_out`, and $w$ is a specific column index inside the range of `W_out`. 

* $\frac{1}{f_H \cdot f_W} \cdot$: Our gradient coefficient, the local derivative will always be this value (will be useful when we make another mask in the implementation). 

* $\mathbb{I}\left( (h', w') \in \Omega_{h, w} \right)$: The absolute input pixel coordinate $(h', w')$ inside the sliding window box that produced output pixel $(h, w)$. 
    * The indicator function this time checks a specific input coordinate and then will "check" if we're in the right window. If we are, we'll perform the computation, otherwise we'll ignore the pixel by multiplying the value by 0. 

## Implementation of our *Fast* Average Pooling Pass

We'd like to keep the `patches` to create our `dpatches` that will be used later in the computation. Since the code is quite similar to the *fast* max pooling pass this section will be a little brief. 

After assuming our patches, we can create a binary mask again. Although we could map this mask to $1 / (fH * fW)$ we can instead set the tensor to all ones to start. After creating the tensor filled with ones, we'll perform the same calculation with dpatches, but perform the calculation in the same line. Another major thing to note is accounting for our padding logic. For this step, we'll move the padding logic outside of the four branches, then each branch will call for their needed sections. The code would be as follows: 

```python
if self.padding == "valid":
    pad_top = pad_bottom = pad_left = pad_right = 0
    padded_H, padded_W = H_in, W_in
elif self.padding == "same":
        
    pad_top = self.pad_top
    pad_bottom = self.pad_bottom
    pad_left = self.pad_left
    pad_right = self.pad_right

    padded_H = H_in + pad_top + pad_bottom
    padded_W = W_in + pad_left + pad_right

if self.pooling_type == "average":
    if self.filter_size == self.strides:

        # code for patches
        # Make dvalues match the shape for the patches memory view

        expanded_dvalues = dvalues[:, :, :, cp.newaxis, cp.newaxis, :]
        mask = cp.ones_like(patches)
        dpatches = (expanded_dvalues * mask) / (fH * fW) 
        
```

Since we already have the padding logic down, we'll create our `dinputs_padded`. Next, we'd like to reduce the dimensions from 6D down to 4D. We'll have to use the same trick from before and transpose the filter height and window width with each other. We do this such that when we reshape after, the row axes line up with each other, and the column axes line up with each other. After reshaping, we can then copy the results onto our `dinputs_padded` tensor. Finally, 

```python

dinputs_padded = cp.zeros((S, padded_H, padded_W, C), dtype=dvalues.dtype)
dinputs_canvas = dpatches.transpose(0, 1, 3, 2, 4, 5).reshape(S, H_out * fH, W_out * fW, C)

dinputs_padded[:, :H_out * fH, :W_out * fW, :] = dinputs_canvas
dinputs = dinputs_padded[:, pad_top : pad_top + H_in, pad_left : pad_left + W_in, :]
```
The code for this section is below:

In [ ]:
from cupy.lib.stride_tricks import as_strided
def backward(self, dvalues):
    
    S, H_out, W_out, C = dvalues.shape
    fH, fW, sH, sW = self.filter_size, self.strides
    _, H_in, W_in, _ = self.inputs.shape

    if self.padding == "valid":
        pad_top = pad_bottom = pad_left = pad_right = 0
        padded_H, padded_W = H_in, W_in
    elif self.padding == "same":
            
        pad_top = self.pad_top
        pad_bottom = self.pad_bottom
        pad_left = self.pad_left
        pad_right = self.pad_right

        padded_H = H_in + pad_top + pad_bottom
        padded_W = W_in + pad_left + pad_right

    if self.pooling_type == "max":
        if self.strides == self.filter_size:
            pass
        if self.strides != self.filter_size:
            pass
        
    if self.pooling_type == "average":
        if self.strides == self.filter_size:
            
            inputs = self.inputs
            patches = as_strided(inputs, 
                                 shape = (S, H_out, W_out, fH, fW, C),
                                 strides = (
                                    inputs.strides[0],      # Step between samples
                                    inputs.strides[1] * sH, # Step between rows
                                    inputs.strides[2] * sW, # Step between columns
                                    inputs.strides[1],      # Move down 1 row inside patch
                                    inputs.strides[2],      # Move right 1 col inside patch
                                    inputs.strides[3],      # Step between each channel
                ))
            
            mask = cp.ones_like(patches)
            expanded_dvalues = dvalues[:, :, :, cp.newaxis, cp.newaxis, :]
            dpatches = (expanded_dvalues * mask) / (fH * fW)

            dinputs_padded = cp.zeros((S, padded_H, padded_W, C), dtype=dvalues.dtype)
            dinputs_canvas = dpatches.transpose(0, 1, 3, 2, 4, 5).reshape(S, H_out * fH, W_out * fW, C)
            dinputs_padded[:, :H_out * fH, :W_out * fW, :] = dinputs_canvas
            dinputs = dinputs_padded[:, pad_top : pad_top + H_in, pad_left : pad_left + W_in, :]
            
        if self.strides != self.filter_size: 
            pass
    return dinputs

# What about the General Case for Average Pooling? 

The current implementation for the general case of average pooling is too slow for reasons presented at the top of the notebook, we'll have to find a faster approach that accounts for overlapping gradients. If we remember the proof for the transposed convolutions, this paradigm can be applied ot our general average pooling logic as well.

Even though average pooling contains no weights (as in no kernels) we can pretend that we have a kernel that we'd like to "flip". Each value in the kernels will actually have the same scale factor $ (1.0 / (fH \times fW))$. Since this matrix filled with uniform values is completely symetrical, we can also avoid flipping the kernel (as doing so creates no difference). Also, because they all contain the same values, we can avoid using GEMM operations using `cp.tensordot`.

### But why Avoid using GEMM Supported Operations? 

Even though GEMM engines like `cuBLAS` and `rocBLAS` are increadibly fast for compute-heavy operations, they are best suited for matrix operations that are contiguous. Due to the limitations of implementing our project via CuPy, we are forced to use `cp.as_strided()` which will result in a non-contiguous viewing of the tensor. The calculation will still work, but under the hood the entire 6D matrix will be copied into VRAM, which is a major issue for keeping our code lean. In this, out VRAM footprint is increased by a factor of $fH \times fW$. To circumvent this, we'll use a operation that supports non-contiguous memory viewings.
* The GPU will use `cp.sum` which supports non-contiguous memory calculations. This avoids the memory bottleneck. 
* Using `cp.sum` is also perfectly reasonable since our average pooling operations need few FLOPs to perform the calculation. Because of this, **average pooling is a bandwidth starved problem, not a compute starved problem**.  

# Implementation of our *General* Average Pass

Like always, we'll have a few assumptions about what we pass in. For this, we'll assume that padding logic `pad_*`, `padded_H` `padded_W` are all saved values, along with `dvalues.shape`, `self.filter_size`, and `self.strides` being saved. We'll have to code logic for a dialated stride (whenever `sH >1 or sW > 1`), the lgoic for our patches, our padded `dvalues`, and finally the logic needed to calculate the average over the scale factor $(1.0 / (fH \times fW))$

Revisiting the code of our updated convolution backwards pass yeilds us with logic for our stride. We can copy the formula below and know how much to dilate each value inside `dvalues` by:

$$
\text{dilated stride} = (H_{out} - 1) \times sH + 1
$$

Next, we can use the slicing notation `[start:stop:step]`, if we write [::sH] we'll start at index 0, end at whatever our height dimension is, but skip ahead by `sH` steps each time. The same will apply for `sW`. 

```python 
dilated_H = (H_out - 1) * sH + 1
dilated_W = (W_out - 1) * sW + 1

dvalues_dilated = cp.zeros(S, dilated_H, dilated_W, C_out, dtype=dvalues.dtype)
```

We can again reference our padding logic from the notebook going over the implementation behind the convolutional backwards pass. There are two cases in padding that we'll account for:

- Case A: `valid` Padding (Forward padding `P` = 0)
    We can use the formula below and plug in P = 0
        $$
    P_b = (fH - 1) - 0 = fH - 1
        $$

- Case B: `same` Padding (Forward padding `P` > 0)
    Using that same formula, we'd arrive at the result:
        $$
    P_b = (fH - 1) - P_f
        $$
    Because the forward pass already had padded the image keeping $h_{out}$ and $h_{in}$ the same, our backwrads pass now needs les padding such that the output **loses** size. 

Now that we have the formulas needed, we can use a simple if else block to handle the logic. We need to still adjust our padding for uneven padding. After arriving at our `backward_pad_*`, we can then use `cp.pad` method to pad our height and width dimensions before performing the computations for `dinputs`. 

```python
# given the forward padding values from before

backward_pad_top = (fH - 1) - pad_top
backward_pad_left = (fW - 1) - pad_left
backward_pad_bottom = (H_in + fH - 1) - dilated_H - backward_pad_top
backward_pad_right = (W_in + fW - 1) - dilated_W - backward_pad_left

dvalues_padded = cp.pad(dvalues_dilated, pad_width =(
    (0, 0), (backward_pad_top, backward_pad_bottom), (backward_pad_left, backward_pad_right), (0, 0)))
```

We have now dilated and padded `dvalues` meaning we're ready for calculation. We'll use patches again to create our views into the array, then we can perform a computation using `patches` and the scale factor to arrive at our result:

```python
self.patches = as_strided(
            dvalues_padded,
            shape=(S, H_out, W_out, fH, fW, C),
            strides=(
                dvalues_padded.strides[0],       # step between samples
                dvalues_padded.strides[1] * sH,  # step down a row
                dvalues_padded.strides[2] * sW,  # step across a column
                dvalues_padded.strides[1],       # move down 1 row inside patch
                dvalues_padded.strides[2],       # move right 1 col inside patch
                dvalues_padded.strides[3],       # step across channels
            )
        )

scale_factor = cp.array(1.0 / (fH * fW), dtype=dvalues.dtype)
dinputs = dvalues_patches.sum(axis=(3,4))* scale_factor
```

We now have created an better vectorized version of the average pass. The full average pass is below: 

In [ ]:
def backward(self, dvalues):

    S, H_out, W_out, C = dvalues.shape
    fH, fW, sH, sW = self.filter_size, self.strides
    _, H_in, W_in, _ = self.inputs.shape

    if self.padding == "valid":
        pad_top = pad_bottom = pad_left = pad_right = 0
        padded_H, padded_W = H_in, W_in
    elif self.padding == "same":
            
        pad_top = self.pad_top
        pad_bottom = self.pad_bottom
        pad_left = self.pad_left
        pad_right = self.pad_right

        padded_H = H_in + pad_top + pad_bottom
        padded_W = W_in + pad_left + pad_right

    if self.pooling_type == "max":
        if self.strides == self.filter_size:
            pass
        if self.strides != self.filter_size:
            pass
        
    if self.pooling_type == "average":
        if self.strides == self.filter_size:
            pass
        if self.strides != self.filter_size:

            dilated_H = (H_out - 1) * sH + 1
            dilated_W = (W_out - 1) * sW + 1

            dvalues_dilated = cp.zeros(S, dilated_H, dilated_W, C, dtype=dvalues.dtype)

            backward_pad_top = (fH - 1) - pad_top
            backward_pad_left = (fW - 1) - pad_left
            backward_pad_bottom = (H_in + fH - 1) - dilated_H - backward_pad_top
            backward_pad_right = (W_in + fW - 1) - dilated_W - backward_pad_left

            dvalues_padded = cp.pad(dvalues_dilated, pad_width = (
                (0, 0), (backward_pad_top, backward_pad_bottom), (backward_pad_left, backward_pad_right), (0, 0)
            ))

            dvalues_patches = as_strided(
                dvalues_padded,
                shape=(S, H_out, W_out, fH, fW, C),
                strides=(
                    dvalues_padded.strides[0],       # step between samples
                    dvalues_padded.strides[1] * sH,  # step down a row
                    dvalues_padded.strides[2] * sW,  # step across a column
                    dvalues_padded.strides[1],       # move down 1 row inside patch
                    dvalues_padded.strides[2],       # move right 1 col inside patch
                    dvalues_padded.strides[3],       # step across channels
                ))
            scale_factor = cp.array(1.0 / (fH * fW), dtype = dvalues.dtype)
            dinputs = dvalues_patches.sum(axis = (3,4) * scale_factor)

            return dinputs

In [ ]:

class Pooling:

    def __init__(self, filter_size = (2, 2), strides = (2, 2),
                  padding = "valid", pooling_type = "max"):
        self.filter_size = filter_size
        self.strides = strides
        self.padding = padding
        self.pooling_type = pooling_type
        
    def forward(self, inputs, training):
        #Inputs should be of shape (S, H_in, W_in, C = D_in) 
        inputs = inputs.astype(cp.float32, copy = False)
        if inputs.ndim != 4:
            raise ValueError(f"Expected a 4D tensor, got {inputs.ndim} instead.")
        S, H_in, W_in, C = inputs.shape
        fH, fW = self.filter_size
        sH, sW = self.strides

        padding = self.padding
        if padding == "valid":
            H_out = int(cp.floor((H_in - fH) / sH + 1).item())
            W_out = int(cp.floor((W_in - fW) / sW + 1).item())

            self.pad_top, self.pad_bottom, self.pad_left, self.pad_right = 0, 0, 0, 0
        elif padding == "same":
            
            H_out = int(cp.ceil(H_in / sH).item())
            W_out = int(cp.ceil(W_in / sW).item())

            pad_h = max((H_out - 1) * sH + fH - H_in, 0)
            pad_w = max((W_out - 1) * sW + fW - W_in, 0)
            self.pad_top = pad_h // 2
            self.pad_bottom = pad_h - self.pad_top
            self.pad_left = pad_w // 2
            self.pad_right = pad_w - self.pad_left
            inputs = cp.pad(inputs, ((0,0), (self.pad_top,self.pad_bottom), (self.pad_left,self.pad_right), (0,0)), mode='constant')
        else: 
            raise ValueError(f"Expected padding == valid or same, recieved {padding} instead")

        #cast our output dimensions into ints from floats. 
        H_out, W_out = int(H_out), int(W_out)

        #create output tensor with new sizes
        self.output = cp.zeros(shape = (S, H_out, W_out, C))
        self.inputs = inputs
        patches = as_strided(
            inputs,
            shape = (S, H_out, W_out, fH, fW, C), 
            strides = (
                inputs.strides[0],      #step between samples
                inputs.strides[1] * sH, #step between rows
                inputs.strides[2] * sW, #step between columns
                inputs.strides[1],      #Move down 1 row inside patch
                inputs.strides[2],      #move right 1col inside patch
                inputs.strides[3],      #step between each channel
            )
        )

        if self.pooling_type == "max":
            pooled = patches.max(axis = (3, 4)) 
            #We'll reshape the window to become a 1d array of size fH * fW
            patches_reshaped = patches.reshape(S, H_out, W_out, fH * fW, C)
            flat_indicies = patches_reshaped.argmax(axis = 3)

            #Now, we'll convert those flat indicies back to row col coordinates withing each
            #(fH, fW) patch
            max_rows, max_cols = cp.unravel_index(flat_indicies, (fH, fW)) 
            self.max_indicies = (max_rows, max_cols) 
        
        elif self.pooling_type == "average":
            pooled = patches.mean(axis = (3, 4))
            
        #Store both of these for backprop
        self.inputs = inputs
        output = pooled
        return output
    
    def backward(self, dvalues):

        S, H_out, W_out, C = dvalues.shape
        fH, fW, sH, sW = self.filter_size, self.strides
        _, H_in, W_in, _ = self.inputs.shape

        if self.padding == "valid":
            pad_top = pad_bottom = pad_left = pad_right = 0
            padded_H, padded_W = H_in, W_in
        elif self.padding == "same":
                
            pad_top = self.pad_top
            pad_bottom = self.pad_bottom
            pad_left = self.pad_left
            pad_right = self.pad_right

            padded_H = H_in + pad_top + pad_bottom
            padded_W = W_in + pad_left + pad_right

        if self.pooling_type == "max":
            if self.strides == self.filter_size:    # Non-overlapping windows
                
                dinputs_padded = cp.zeros((S, padded_H, padded_W, C), dtype=dvalues.dtype)
                #compute non-overlapping windows
                inputs = self.inputs
                patches = as_strided(
                    inputs,
                    shape = (S, H_out, W_out, fH, fW, C), 
                    strides = (
                        inputs.strides[0],      # Step between samples
                        inputs.strides[1] * sH, # Step between rows
                        inputs.strides[2] * sW, # Step between columns
                        inputs.strides[1],      # Move down 1 row inside patch
                        inputs.strides[2],      # Move right 1 col inside patch
                        inputs.strides[3],      # Step between each channel
                    )
                )
                expanded_output = self.output[:, :, :, cp.newaxis, cp.newaxis, :]
                mask = (patches == expanded_output)

                #expand upstream gradient 
                expanded_dvalues = dvalues[:, :, :, cp.newaxis, cp.newaxis, :]
                dpatches = expanded_dvalues * mask

                dinputs_canvas = dpatches.transpose(0, 1, 3, 2, 4, 5).reshape(S, H_out * fH, W_out * fW, C)
                
                dinputs_padded[:, :H_out * fH, :W_out * fW, :] = dinputs_canvas
                dinputs = dinputs_padded[:, pad_top : pad_top + H_in, pad_left : pad_left + W_in, :]

            else:                                   # Branch where windows overlap
                
                max_rows, max_cols = self.max_indicies

                s_idx, h_idx, w_idx, c_idx = cp.ogrid[:S, :H_out, :W_out, :C]

                input_h = (h_idx * sH) + max_rows
                input_w = (w_idx * sW) + max_cols

                cp.add.at(dinputs_padded, (s_idx, input_h, input_w, c_idx), dvalues)

                dinputs = dinputs_padded[:, 
                            pad_top : pad_top + H_in, 
                            pad_left : pad_left + W_in, :]       
        if self.pooling_type == "average":
            if self.strides == self.filter_size: # Non-overlapping windows
                inputs = self.inputs
                patches = as_strided(inputs, 
                                    shape = (S, H_out, W_out, fH, fW, C),
                                    strides = (
                                        inputs.strides[0],      # Step between samples
                                        inputs.strides[1] * sH, # Step between rows
                                        inputs.strides[2] * sW, # Step between columns
                                        inputs.strides[1],      # Move down 1 row inside patch
                                        inputs.strides[2],      # Move right 1 col inside patch
                                        inputs.strides[3],      # Step between each channel
                    ))
                
                mask = cp.ones_like(patches)
                expanded_dvalues = dvalues[:, :, :, cp.newaxis, cp.newaxis, :]
                dpatches = (expanded_dvalues * mask) / (fH * fW)

                dinputs_padded = cp.zeros((S, padded_H, padded_W, C), dtype=dvalues.dtype)
                dinputs_canvas = dpatches.transpose(0, 1, 3, 2, 4, 5).reshape(S, H_out * fH, W_out * fW, C)
                dinputs_padded[:, :H_out * fH, :W_out * fW, :] = dinputs_canvas
                dinputs = dinputs_padded[:, pad_top : pad_top + H_in, pad_left : pad_left + W_in, :]
                
            else:                                   # Branch where windows overlap

                dilated_H = (H_out - 1) * sH + 1
                dilated_W = (W_out - 1) * sW + 1

                dvalues_dilated = cp.zeros(S, dilated_H, dilated_W, C, dtype=dvalues.dtype)

                backward_pad_top = (fH - 1) - pad_top
                backward_pad_left = (fW - 1) - pad_left
                backward_pad_bottom = (H_in + fH - 1) - dilated_H - backward_pad_top
                backward_pad_right = (W_in + fW - 1) - dilated_W - backward_pad_left

                dvalues_padded = cp.pad(dvalues_dilated, pad_width = (
                    (0, 0), (backward_pad_top, backward_pad_bottom), (backward_pad_left, backward_pad_right), (0, 0)
                ))

                dvalues_patches = as_strided(
                    dvalues_padded,
                    shape=(S, H_out, W_out, fH, fW, C),
                    strides=(
                        dvalues_padded.strides[0],       # step between samples
                        dvalues_padded.strides[1] * sH,  # step down a row
                        dvalues_padded.strides[2] * sW,  # step across a column
                        dvalues_padded.strides[1],       # move down 1 row inside patch
                        dvalues_padded.strides[2],       # move right 1 col inside patch
                        dvalues_padded.strides[3],       # step across channels
                    ))
                scale_factor = cp.array(1.0 / (fH * fW), dtype = dvalues.dtype)
                dinputs = dvalues_patches.sum(axis = (3,4) * scale_factor)

        return dinputs